# Divye FE — 3-Way Interactions

Extends `s6e2-divye-fe.ipynb` with **3-way multiplicative interactions** of the top TE features.

Base divye FE has pairwise (2-way) products of top-8 TE features (28 features).  
This adds all C(8,3)=56 three-way products on top, giving the model joint risk signals like:

`thallium_te × chest_pain_type_te × number_of_vessels_fluro_te`

= approximate probability that all three high-risk factors are simultaneously present.

2-way interactions capture pairs; 3-way captures the compounding effect of multiple risk factors together.

Baseline: catboost_divye_fe OOF=0.95566  LB=0.95381

In [1]:
import subprocess
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


In [2]:
def compute_freq_features(train_df, test_df, cols):
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq


def compute_oof_te(train_df, test_df, cols, y, skf):
    global_mean = y.mean()
    oof_te  = {f'{col}_te': np.zeros(len(train_df)) for col in cols}
    test_te = {f'{col}_te': np.zeros(len(test_df))  for col in cols}
    for col in cols:
        key, tr_vals, te_vals, fold_test = f'{col}_te', train_df[col].values, test_df[col].values, []
        for tr_idx, val_idx in skf.split(train_df, y):
            te_map = pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]}).groupby('v')['t'].mean()
            oof_te[key][val_idx] = pd.Series(tr_vals[val_idx]).map(te_map).fillna(global_mean).values
            fold_test.append(pd.Series(te_vals).map(te_map).fillna(global_mean).values)
        test_te[key] = np.mean(fold_test, axis=0)
    return oof_te, test_te


def make_2way_interactions(te_dict, top_cols):
    return {f'{c1}_x_{c2}': te_dict[c1] * te_dict[c2]
            for c1, c2 in combinations(top_cols, 2)}


def make_3way_interactions(te_dict, top_cols):
    return {f'{c1}_x_{c2}_x_{c3}': te_dict[c1] * te_dict[c2] * te_dict[c3]
            for c1, c2, c3 in combinations(top_cols, 3)}


def build_augmented(base_df, *dicts):
    df = base_df.copy().reset_index(drop=True)
    for d in dicts:
        for name, vals in d.items():
            df[name] = vals
    return df


print('Frequency encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('OOF target encoding...')
oof_te, test_te = compute_oof_te(X, X_test, ALL_FEATURES, y, SKF)

corrs = {col: abs(np.corrcoef(vals, y)[0, 1]) for col, vals in oof_te.items()}
top8  = sorted(corrs, key=corrs.get, reverse=True)[:8]
print(f'Top-8 TE features: {top8}')

tr_2way = make_2way_interactions(oof_te,  top8)   # 28 features
te_2way = make_2way_interactions(test_te, top8)
tr_3way = make_3way_interactions(oof_te,  top8)   # 56 features
te_3way = make_3way_interactions(test_te, top8)

print(f'2-way interactions: {len(tr_2way)}')
print(f'3-way interactions: {len(tr_3way)}')
print(f'Total new features: {len(tr_freq) + len(oof_te) + len(tr_2way) + len(tr_3way)}')

Frequency encoding...
OOF target encoding...
Top-8 TE features: ['thallium_te', 'chest_pain_type_te', 'max_hr_te', 'number_of_vessels_fluro_te', 'st_depression_te', 'exercise_angina_te', 'slope_of_st_te', 'sex_te']
2-way interactions: 28
3-way interactions: 56
Total new features: 110


In [3]:
CATBOOST_PARAMS = dict(
    iterations        = 2000,
    learning_rate     = 0.05,
    depth             = 6,
    task_type         = 'CPU',
    thread_count      = -1,
    cat_features      = CAT_FEATURES,
    random_seed       = 42,
    verbose           = 0,
)

global_mean = y.mean()
oof_preds   = np.zeros(len(y))
test_folds  = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')
    X_tr_base  = X.iloc[tr_idx].reset_index(drop=True)
    X_val_base = X.iloc[val_idx].reset_index(drop=True)
    y_tr, y_val = y[tr_idx], y[val_idx]

    fold_tr_freq, fold_te_freq  = compute_freq_features(X_tr_base, X_test,    ALL_FEATURES)
    _,            fold_val_freq = compute_freq_features(X_tr_base, X_val_base, ALL_FEATURES)

    fold_tr_te, fold_val_te, fold_te_te = {}, {}, {}
    for col in ALL_FEATURES:
        key    = f'{col}_te'
        te_map = pd.DataFrame({'v': X_tr_base[col].values, 't': y_tr}).groupby('v')['t'].mean()
        fold_tr_te[key]  = X_tr_base[col].map(te_map).fillna(global_mean).values
        fold_val_te[key] = X_val_base[col].map(te_map).fillna(global_mean).values
        fold_te_te[key]  = X_test[col].map(te_map).fillna(global_mean).values

    fold_tr_2way  = make_2way_interactions(fold_tr_te,  top8)
    fold_val_2way = make_2way_interactions(fold_val_te, top8)
    fold_te_2way  = make_2way_interactions(fold_te_te,  top8)

    fold_tr_3way  = make_3way_interactions(fold_tr_te,  top8)
    fold_val_3way = make_3way_interactions(fold_val_te, top8)
    fold_te_3way  = make_3way_interactions(fold_te_te,  top8)

    X_tr_aug  = build_augmented(X_tr_base,  fold_tr_freq,  fold_tr_te,  fold_tr_2way,  fold_tr_3way)
    X_val_aug = build_augmented(X_val_base, fold_val_freq, fold_val_te, fold_val_2way, fold_val_3way)
    X_te_aug  = build_augmented(X_test,     fold_te_freq,  fold_te_te,  fold_te_2way,  fold_te_3way)

    m = CatBoostClassifier(**CATBOOST_PARAMS)
    m.fit(X_tr_aug, y_tr, eval_set=(X_val_aug, y_val), early_stopping_rounds=50)

    oof_preds[val_idx] = m.predict_proba(X_val_aug)[:, 1]
    test_folds[fold]   = m.predict_proba(X_te_aug)[:, 1]
    print(f'Fold {fold+1}  AUC={roc_auc_score(y_val, oof_preds[val_idx]):.5f}  best_iter={m.best_iteration_}')

cv_auc     = roc_auc_score(y, oof_preds)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (divye_fe_3way): {cv_auc:.5f}')
print(f'Baseline divye_fe:       0.95566')
print(f'Delta:                   {cv_auc - 0.95566:+.5f}')


=== Fold 1/5 ===


/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name]

Fold 1  AUC=0.95601  best_iter=568

=== Fold 2/5 ===


/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name]

Fold 2  AUC=0.95486  best_iter=501

=== Fold 3/5 ===


...

Fold 3  AUC=0.95569  best_iter=596

=== Fold 4/5 ===


/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name]

Fold 4  AUC=0.95527  best_iter=611

=== Fold 5/5 ===


/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1276684/3096682388.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name]

Fold 5  AUC=0.95618  best_iter=539

OOF AUC (divye_fe_3way): 0.95560
Baseline divye_fe:       0.95566
Delta:                   -0.00006


In [4]:
np.save('submissions/oof_divye_fe_3way.npy', oof_preds)

label = 'catboost_divye_fe_3way'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc={cv_auc:.4f}'
print(f'Saved: {fname}')
print(f'desc:  {desc}')
print(f'OOF AUC: {cv_auc:.5f}  (baseline: 0.95566  delta: {cv_auc - 0.95566:+.5f})')

Saved: submissions/catboost_divye_fe_3way.csv
desc:  catboost_divye_fe_3way | cv_auc=0.9556
OOF AUC: 0.95560  (baseline: 0.95566  delta: -0.00006)


In [5]:
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

catboost_divye_fe_3way: submitted
